# Assignment 1: training pipeline (credit risk)

**Decision supported:** at application time, should the bank **approve or reject** a loan?
The model estimates the probability that the applicant **defaults** (`loan_status = 1`),
and we reject when that probability is above a threshold chosen to **minimise money lost**.

Pipeline: load & clean → split → engineer features (fit on train only) → train the same
L2-regularised logistic regression three ways → compare with a naive baseline → pick the
threshold on validation → report on test → save everything the app needs.

In [ ]:
import json

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

from credit_pipeline import (ARTIFACTS, SEED, TARGET, CreditFeatures,
                             LogisticRegressionModule, business_cost,
                             load_clean_data)

np.random.seed(SEED)
torch.manual_seed(SEED)
ARTIFACTS.mkdir(exist_ok=True)

## 1. Load and clean
Drop exact duplicates (they could land in both train and test) and impossible ages.
Impossible employment lengths are handled inside the feature step (set to missing).

In [ ]:
df = load_clean_data()
print(f"{len(df):,} rows after cleaning, default rate = {df[TARGET].mean():.1%}")

## 2. Split: 60 / 20 / 20, stratified
The dataset has **no application date**, so a time-based split is impossible; a stratified
random split keeps the ~22% default rate equal in every split.
*Validation* is used to choose C, the learning rates and the decision threshold;
*test* is touched only once, for the final numbers.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.4, stratify=df[TARGET], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df[TARGET], random_state=SEED)

split = pd.Series("train", index=df.index)
split[val_df.index] = "val"
split[test_df.index] = "test"
split.rename("split").to_frame().to_csv(ARTIFACTS / "split.csv", index_label="row")
print({name: len(part) for name, part in [("train", train_df), ("val", val_df), ("test", test_df)]})

## 3. Features (fitted on the training split only)

In [ ]:
features = CreditFeatures().fit(train_df)
X_train, X_val, X_test = (features.transform(d) for d in (train_df, val_df, test_df))
y_train, y_val, y_test = (d[TARGET].to_numpy() for d in (train_df, val_df, test_df))
n_train, n_features = X_train.shape
print(n_features, "features:", features.feature_names_)

## 4a. Naive baseline
Predict the training default rate for everyone. As a *decision* this means
"approve everybody", which is what the bank would do without a model.

In [ ]:
base_rate = y_train.mean()

## 4b. scikit-learn, and choosing the regularisation strength C
sklearn minimises `C * sum(log-loss) + ½‖w‖²` (intercept not penalised).
Small grid, scored on validation log loss, the metric we care about because the threshold
step needs *well-calibrated probabilities*, not just a good ranking.

In [ ]:
C_GRID = [0.001, 0.01, 0.1, 1.0, 10.0]
c_scores = {}
for C in C_GRID:
    m = LogisticRegression(C=C, max_iter=5000).fit(X_train, y_train)
    c_scores[C] = log_loss(y_val, m.predict_proba(X_val)[:, 1])
# Log loss is almost flat for C >= 0.1, so rather than taking the edge of the grid we take the
# STRONGEST regularisation (smallest C) whose val log loss is within 0.001 of the best one.
best_loss = min(c_scores.values())
best_C = min(C for C, v in c_scores.items() if v <= best_loss + 0.001)
print("val log loss per C:", {k: round(v, 4) for k, v in c_scores.items()}, "-> best C =", best_C)

sk_model = LogisticRegression(C=best_C, max_iter=5000).fit(X_train, y_train)
joblib.dump(sk_model, ARTIFACTS / "sklearn_logreg.joblib")

# The SAME objective written per-sample (divide by C*n): mean log-loss + (λ/2)‖w‖²
lam = 1.0 / (best_C * n_train)
print(f"equivalent L2 strength for PyTorch: lambda = {lam:.2e}")

## 4c. Manual PyTorch loop (raw tensors + autograd + hand-written update)
Full-batch gradient descent on exactly the sklearn objective, so in theory it should
converge to the same weights.

In [ ]:
Xtr_t, ytr_t = torch.tensor(X_train), torch.tensor(y_train, dtype=torch.float32)
Xva_t = torch.tensor(X_val)
Xte_t = torch.tensor(X_test)


def manual_loss(w, b, X, y):
    z = X @ w + b
    # log-loss written out: -[y·log σ(z) + (1-y)·log(1-σ(z))], using log σ(-z) = log(1-σ(z))
    nll = -(y * F.logsigmoid(z) + (1 - y) * F.logsigmoid(-z)).mean()
    return nll + 0.5 * lam * (w ** 2).sum()   # bias is not penalised, like sklearn


def train_manual(lr, epochs=3000):
    w = torch.zeros(n_features, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    history = []
    for epoch in range(epochs):
        loss = manual_loss(w, b, Xtr_t, ytr_t)
        loss.backward()                      # autograd fills w.grad and b.grad
        with torch.no_grad():                # the update itself must not be tracked
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        if epoch % 50 == 0:
            history.append(loss.item())
    return w.detach(), b.detach(), history


def manual_predict(w, b, X):
    return torch.sigmoid(X @ w + b).numpy()


LR_GRID_MANUAL = [0.1, 0.5, 1.0, 2.0]
manual_runs = {}
for lr in LR_GRID_MANUAL:
    w, b, hist = train_manual(lr)
    manual_runs[lr] = (w, b, hist, log_loss(y_val, manual_predict(w, b, Xva_t)))
best_lr_manual = min(manual_runs, key=lambda k: manual_runs[k][3])
w_manual, b_manual, hist_manual, _ = manual_runs[best_lr_manual]
print("val log loss per lr:", {k: round(v[3], 4) for k, v in manual_runs.items()}, "-> best lr =", best_lr_manual)
torch.save({"w": w_manual, "b": b_manual}, ARTIFACTS / "manual_torch.pt")

## 4d. Standard PyTorch workflow (nn.Module + DataLoader + torch.optim)
Mini-batches of 256 and `optim.SGD`. The L2 penalty is `weight_decay` on the weights only
(a separate parameter group with no decay for the bias), which adds λ·w to the
gradient: the gradient of (λ/2)‖w‖², i.e. the same penalty as above.

In [ ]:
loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=256, shuffle=True,
                    generator=torch.Generator().manual_seed(SEED))


def train_standard(lr, epochs=60):
    torch.manual_seed(SEED)
    model = LogisticRegressionModule(n_features)
    optimizer = torch.optim.SGD([
        {"params": [model.linear.weight], "weight_decay": lam},
        {"params": [model.linear.bias], "weight_decay": 0.0},
    ], lr=lr)
    loss_fn = torch.nn.BCEWithLogitsLoss()
    history = []
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            full = loss_fn(model(Xtr_t), ytr_t) + 0.5 * lam * (model.linear.weight ** 2).sum()
        history.append(float(full))
    return model, history


def standard_predict(model, X):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(X)).numpy()


LR_GRID_STANDARD = [0.01, 0.05, 0.1, 0.5]
standard_runs = {}
for lr in LR_GRID_STANDARD:
    model, hist = train_standard(lr)
    standard_runs[lr] = (model, hist, log_loss(y_val, standard_predict(model, Xva_t)))
best_lr_standard = min(standard_runs, key=lambda k: standard_runs[k][2])
std_model, hist_standard, _ = standard_runs[best_lr_standard]
print("val log loss per lr:", {k: round(v[2], 4) for k, v in standard_runs.items()}, "-> best lr =", best_lr_standard)
torch.save(std_model.state_dict(), ARTIFACTS / "torch_module.pt")

## 5. Decision threshold (chosen on validation, by business cost)
Cost of approving a defaulter = LGD × loan amount; cost of rejecting a good customer =
profit margin × loan amount. We pick, for each model, the threshold that minimises total
cost on the validation set.

In [ ]:
val_probs = {
    "baseline": np.full(len(y_val), base_rate),
    "sklearn": sk_model.predict_proba(X_val)[:, 1],
    "manual_torch": manual_predict(w_manual, b_manual, Xva_t),
    "torch_module": standard_predict(std_model, Xva_t),
}
thresholds = np.round(np.arange(0.01, 1.00, 0.01), 2)
best_thresholds = {}
for name, p in val_probs.items():
    if name == "baseline":
        best_thresholds[name] = 1.0   # "approve everybody"
        continue
    costs = [business_cost(y_val, p, val_df["loan_amnt"], t)["total_cost"] for t in thresholds]
    best_thresholds[name] = float(thresholds[int(np.argmin(costs))])
print("cost-minimising thresholds (val):", best_thresholds)

## 6. Final comparison on the test set

In [ ]:
test_probs = {
    "baseline": np.full(len(y_test), base_rate),
    "sklearn": sk_model.predict_proba(X_test)[:, 1],
    "manual_torch": manual_predict(w_manual, b_manual, Xte_t),
    "torch_module": standard_predict(std_model, Xte_t),
}
rows = []
for name, p in test_probs.items():
    t = best_thresholds[name]
    cost = business_cost(y_test, p, test_df["loan_amnt"], t)
    rejected = cost["tp"] + cost["fp"]
    rows.append({
        "model": name,
        "roc_auc": roc_auc_score(y_test, p) if name != "baseline" else 0.5,
        "log_loss": log_loss(y_test, p),
        "threshold": t,
        "recall_defaults": cost["tp"] / (cost["tp"] + cost["fn"]),
        "precision_reject": cost["tp"] / rejected if rejected else float("nan"),
        "approval_rate": 1 - rejected / len(y_test),
        "total_cost": cost["total_cost"],
    })
results = pd.DataFrame(rows).set_index("model")
print(results.round(4).to_string())

### Do the three implementations agree?

In [ ]:
w_sk = sk_model.coef_.ravel()
w_std = std_model.linear.weight.detach().numpy().ravel()
coefs = pd.DataFrame({"sklearn": w_sk, "manual_torch": w_manual.numpy(), "torch_module": w_std},
                     index=features.feature_names_)
coefs.loc["(intercept)"] = [sk_model.intercept_[0], b_manual.item(), std_model.linear.bias.item()]
agreement = {
    "mean_abs_prob_diff_manual_vs_sklearn": float(np.abs(test_probs["manual_torch"] - test_probs["sklearn"]).mean()),
    "mean_abs_prob_diff_module_vs_sklearn": float(np.abs(test_probs["torch_module"] - test_probs["sklearn"]).mean()),
    "max_abs_prob_diff_manual_vs_sklearn": float(np.abs(test_probs["manual_torch"] - test_probs["sklearn"]).max()),
    "max_abs_prob_diff_module_vs_sklearn": float(np.abs(test_probs["torch_module"] - test_probs["sklearn"]).max()),
    "max_abs_coef_diff_manual_vs_sklearn": float(np.abs(w_manual.numpy() - w_sk).max()),
    "max_abs_coef_diff_module_vs_sklearn": float(np.abs(w_std - w_sk).max()),
}
print(coefs.round(3).to_string())
print(json.dumps(agreement, indent=2))

## 7. Experiment: what if we (wrongly) used the lender's own grade?
Not used by the final models, only to quantify the leakage question discussed in REPORT.md.

In [ ]:
grade_tr = pd.get_dummies(train_df["loan_grade"], dtype=float).reindex(columns=list("BCDEFG"), fill_value=0)
grade_te = pd.get_dummies(test_df["loan_grade"], dtype=float).reindex(columns=list("BCDEFG"), fill_value=0)
leaky = LogisticRegression(C=best_C, max_iter=5000).fit(np.hstack([X_train, grade_tr]), y_train)
leaky_auc = roc_auc_score(y_test, leaky.predict_proba(np.hstack([X_test, grade_te]))[:, 1])
print(f"test AUC with loan_grade: {leaky_auc:.4f} vs without: {results.loc['sklearn', 'roc_auc']:.4f}")

## 8. Save what the app needs (it never retrains)

In [ ]:
joblib.dump(features, ARTIFACTS / "features.joblib")
coefs.to_csv(ARTIFACTS / "coefficients.csv")
summary = {
    "best_C": best_C, "lambda": lam, "c_grid_val_logloss": c_scores,
    "best_lr_manual": best_lr_manual, "best_lr_standard": best_lr_standard,
    "manual_lr_grid_val_logloss": {k: v[3] for k, v in manual_runs.items()},
    "standard_lr_grid_val_logloss": {k: v[2] for k, v in standard_runs.items()},
    "base_rate": float(base_rate), "thresholds": best_thresholds,
    "loss_history": {"manual_torch_every_50_steps": hist_manual, "torch_module_per_epoch": hist_standard},
    "agreement": agreement, "leaky_grade_test_auc": leaky_auc,
    "test_results": results.reset_index().to_dict(orient="records"),
}
(ARTIFACTS / "metrics.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved to", ARTIFACTS)